# Notebook 2: Schema-Aware RAG & SQL Execution Agent Pipeline

Run all cells **on Kaggle with a Tesla T4**. Each chunk's assertion test must pass
before advancing. BGE embeddings run on CPU; Qwen + final LoRA run on one GPU.
The 100 examples, final adapter, base revision and databases match Notebook 1.
No validation gold SQL is provided to retrieval or generation.

Results use strict `pandas.DataFrame.equals`: labels, dtypes and row/column order
matter. This is single-database EX, not official Spider test-suite accuracy.
The measured schema-free **3-epoch** score is 15%; the 60-step score is unmeasured.


## Chunk 1: Environment, Dependencies & ChromaDB Setup


In [ ]:
import os, sys, subprocess
from pathlib import Path
assert Path('/kaggle/input').is_dir(), 'Execute on Kaggle, not locally'
os.environ['HF_HOME'] = '/tmp/hf-rag-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
os.chdir('/kaggle/working')
Path('notebook2').mkdir(exist_ok=True)
Path('notebook2/__init__.py').write_text('')
sys.path.insert(0, '/kaggle/working')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *['chromadb==1.5.9', 'fastembed>=0.7,<1', 'transformers==5.17.0', 'peft==0.21.2', 'tokenizers==0.23.3', 'accelerate>=1.0', 'bitsandbytes==0.50.2', 'safetensors>=0.5', 'sqlglot==30.30.0', 'pandas==2.3.3', 'nvidia-ml-py>=12']], check=True)


In [ ]:
%%writefile /kaggle/working/notebook2/environment.py
"""Chunk 1: persistent Chroma configuration and remote-only verification."""
import importlib.metadata
import json
import logging
import os
from pathlib import Path

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "spider_ddl_bge_small_v1"
VRAM_LIMIT_BYTES = 10_000_000_000  # Decimal GB, stricter than 10 GiB.


def require_kaggle():
    if not Path("/kaggle/input").is_dir() or not Path("/kaggle/working").is_dir():
        raise RuntimeError("Run this notebook on Kaggle with a T4 GPU; local execution is disabled.")


def configure_logging(output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, force=True,
        format="%(asctime)s %(levelname)s %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(output / "evaluation.log")])


def open_collection(directory):
    import chromadb
    from chromadb.config import Settings
    client = chromadb.PersistentClient(path=str(directory),
        settings=Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection(COLLECTION_NAME, embedding_function=None,
        configuration={"hnsw": {"space": "cosine"}},
        metadata={"embedding_model": EMBEDDING_MODEL, "schema_format": "sqlite-ddl-v1"})
    if collection.metadata.get("embedding_model") != EMBEDDING_MODEL:
        raise ValueError("Existing collection uses a different embedding model")
    return client, collection


def verify_environment():
    """Remote assertion test, before any schema or model work."""
    require_kaggle()
    import sqlite3
    import tempfile
    import torch
    import sqlglot
    assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator"
    assert "T4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
    assert sqlglot.parse_one("SELECT 1", read="sqlite") is not None
    with sqlite3.connect(":memory:") as conn:
        assert conn.execute("SELECT 1").fetchone() == (1,)
    with tempfile.TemporaryDirectory() as directory:
        client, collection = open_collection(directory)
        collection.add(ids=["probe"], embeddings=[[1.0, 0.0]], documents=["SELECT 1"],
                       metadatas=[{"db_id": "probe"}])
        _, reopened = open_collection(directory)
        assert reopened.get(where={"db_id": "probe"})["ids"] == ["probe"]
    versions = {name: importlib.metadata.version(name) for name in
        ("torch", "chromadb", "fastembed", "transformers", "peft", "bitsandbytes", "pandas", "sqlglot")}
    logging.info("CHUNK 1 PASS | GPU=%s | versions=%s", torch.cuda.get_device_name(0), versions)
    return versions


In [ ]:
from notebook2.environment import configure_logging, verify_environment
OUTPUT = Path('/kaggle/working/rag_evaluation')
configure_logging(OUTPUT)
versions = verify_environment()
passed_chunks = [1]


## Chunk 2: Schema Extraction & Metadata-Filtered Vector Indexing

One original SQLite DDL document per table, including PK/FK relationships.
Chroma applies `where={"db_id": db_id}` before top-k retrieval. Unknown databases
raise an error. No global fallback. BGE's 512-token embedding window may truncate
very wide DDL for ranking; the full stored DDL is injected into the model prompt.


In [ ]:
%%writefile /kaggle/working/notebook2/schemas.py
"""Chunk 2: per-table SQLite DDL, CPU BGE vectors, strict database filtering."""
from contextlib import closing
import hashlib
import logging
from pathlib import Path
import re
import sqlite3

from notebook2.environment import EMBEDDING_MODEL, open_collection


def database_path(root, db_id):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", db_id):
        raise ValueError("Invalid Spider database ID")
    root = Path(root).resolve()
    path = (root / db_id / f"{db_id}.sqlite").resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise FileNotFoundError(path)
    return path


def extract_schemas(path, db_id):
    """Original CREATE TABLE retains column types, composite PKs and FK clauses.

    PRAGMA metadata supplements DDL and explicitly records key relationships.
    No data values, validation questions, or gold SQL enter the vector index.
    """
    records = []
    with closing(sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True)) as conn:
        tables = conn.execute("SELECT name, sql FROM sqlite_master WHERE type='table' "
                              "AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL ORDER BY name").fetchall()
        for name, ddl in tables:
            quoted = '"' + name.replace('"', '""') + '"'
            columns = conn.execute(f"PRAGMA table_info({quoted})").fetchall()
            keys = conn.execute(f"PRAGMA foreign_key_list({quoted})").fetchall()
            primary = [r[1] for r in sorted(columns, key=lambda r: r[5]) if r[5]]
            lines = [f"-- Database: {db_id}; table: {name}", ddl.rstrip("; ") + ";"]
            if primary:
                lines.append("-- Primary key: " + ", ".join(primary))
            for fk in keys:
                lines.append(f"-- Foreign key {fk[0]}.{fk[1]}: {name}.{fk[3]} -> {fk[2]}.{fk[4] or '(primary key)'}")
            document = "\n".join(lines)
            records.append({"id": hashlib.sha256(f"{db_id}\0{name}".encode()).hexdigest(),
                "document": document, "metadata": {"db_id": db_id, "table_name": name,
                    "schema_sha256": hashlib.sha256(document.encode()).hexdigest()}})
    if not records:
        raise ValueError(f"No tables in {db_id}")
    return records


class SchemaIndex:
    def __init__(self, directory):
        from fastembed import TextEmbedding
        self.client, self.collection = open_collection(directory)
        # ONNX CPU embeddings leave GPU memory exclusively for the language model.
        self.encoder = TextEmbedding(model_name=EMBEDDING_MODEL,
            cache_dir="/tmp/bge-cache", providers=["CPUExecutionProvider"], threads=2)

    def ingest(self, path, db_id):
        records = extract_schemas(path, db_id)
        vectors = [v.tolist() for v in self.encoder.embed([r["document"] for r in records])]
        assert all(len(v) == 384 for v in vectors)
        self.collection.upsert(ids=[r["id"] for r in records], embeddings=vectors,
            documents=[r["document"] for r in records], metadatas=[r["metadata"] for r in records])
        stale = set(self.collection.get(where={"db_id": db_id})["ids"]) - {r["id"] for r in records}
        if stale:
            self.collection.delete(ids=sorted(stale))
        logging.info("Indexed %s: %d tables", db_id, len(records))
        return records

    def retrieve(self, user_query, db_id, top_k=3):
        if not user_query.strip() or top_k < 1:
            raise ValueError("Non-empty question and positive top_k required")
        available = self.collection.get(where={"db_id": db_id})["ids"]
        if not available:
            raise KeyError(f"Database {db_id!r} has not been indexed")
        vector = next(iter(self.encoder.query_embed(user_query))).tolist()
        result = self.collection.query(query_embeddings=[vector],
            where={"db_id": db_id}, n_results=min(top_k, len(available)),
            include=["documents", "metadatas", "distances"])
        matches = [{"document": doc, "metadata": meta, "distance": distance}
            for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])]
        assert matches and all(m["metadata"]["db_id"] == db_id for m in matches), "Cross-database retrieval"
        return "\n\n".join(m["document"] for m in matches), matches

    def retrieve_relevant_schemas(self, user_query: str, db_id: str, top_k: int = 3) -> str:
        return self.retrieve(user_query, db_id, top_k)[0]


_active_index = None


def initialize_schema_index(directory):
    global _active_index
    _active_index = SchemaIndex(directory)
    return _active_index


def retrieve_relevant_schemas(user_query: str, db_id: str, top_k: int = 3) -> str:
    if _active_index is None:
        raise RuntimeError("Call initialize_schema_index first")
    return _active_index.retrieve_relevant_schemas(user_query, db_id, top_k)


def verify_schemas():
    """Real BGE + Chroma assertion test on remote synthetic databases."""
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        index = SchemaIndex(Path(directory) / "chroma")
        for db_id in ("alpha", "beta"):
            path = Path(directory) / f"{db_id}.sqlite"
            with sqlite3.connect(path) as conn:
                conn.executescript("CREATE TABLE parent(id INTEGER PRIMARY KEY, name TEXT);"
                    "CREATE TABLE child(id INTEGER PRIMARY KEY, parent_id INTEGER REFERENCES parent(id));")
            records = index.ingest(path, db_id)
            assert len(records) == 2 and "REFERENCES" in records[0]["document"]
            index.ingest(path, db_id)  # Upserts must be idempotent.
        assert index.collection.count() == 4
        for db_id in ("alpha", "beta"):
            text, matches = index.retrieve("List child and parent names", db_id, 20)
            assert len(matches) == 2 and all(m["metadata"]["db_id"] == db_id for m in matches)
            assert f"Database: {db_id}" in text
        try:
            index.retrieve("parent names", "missing")
            raise AssertionError("Unknown database must not fall back to all databases")
        except KeyError:
            pass
    logging.info("CHUNK 2 PASS | real BGE/Chroma isolation, PK/FK extraction, idempotence")


In [ ]:
from notebook2.schemas import verify_schemas, initialize_schema_index, retrieve_relevant_schemas
verify_schemas()
passed_chunks.append(2)
schema_index = initialize_schema_index('/kaggle/working/chroma_schema_store')


## Chunk 3: Model & LoRA Adapter Loading with Dynamic Prompting

NF4, double quantization, FP16 compute, batch size 1, greedy decoding, 2048-token
context and 256 new tokens. Prompts exceeding the limit are recorded as failures,
never silently truncated. GPU allocator cap: 8.8 GB. Both allocator high-water
marks and sampled NVML usage must remain below **10 decimal GB**.


In [ ]:
%%writefile /kaggle/working/notebook2/model.py
"""Chunk 3: native Qwen chat, NF4 + LoRA, bounded single-example inference."""
import logging
import threading
import time

from notebook2.environment import VRAM_LIMIT_BYTES, require_kaggle

BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
BASE_REVISION = "4858886f896bd05db823557fc5cbc4ac28342af7"
SYSTEM_PROMPT = ("You are an expert SQL assistant. Given the following database schema, "
    "write a valid SQL query that accurately answers the question.\n\n"
    "### Database Schema:\n{retrieved_schema}")


def chat_messages(question, db_id, retrieved_schema):
    if not retrieved_schema.strip():
        raise ValueError("Refusing schema-free inference in Notebook 2")
    return [{"role": "system", "content": SYSTEM_PROMPT.format(retrieved_schema=retrieved_schema)},
            {"role": "user", "content": f"Database: {db_id}\nQuestion: {question}"}]


def assistant_target(query):
    """Reference formatting only; never append a gold target during inference."""
    return {"role": "assistant", "content": f"```sql\n{query}\n```"}


class MemoryBudgetExceeded(RuntimeError):
    pass


class MemoryMonitor:
    """Sample NVML device usage and record exact PyTorch allocator high-water marks.

    NVML includes context and non-PyTorch allocations. Its peak is sampled at
    50 ms; the allocator is additionally capped at 8.8 GB. This is a measured
    budget check, not a promise about every possible unsampled transient.
    """
    def __init__(self, torch):
        import pynvml
        self.torch, self.nvml = torch, pynvml
        pynvml.nvmlInit()
        uuid = str(torch.cuda.get_device_properties(0).uuid)
        self.handle = pynvml.nvmlDeviceGetHandleByUUID(uuid)
        self.peak_device_bytes = 0
        self.error = None
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._watch, daemon=True)
        self.thread.start()

    def _watch(self):
        try:
            while not self.stop_event.is_set():
                self.sample()
                self.stop_event.wait(0.05)
        except Exception as exc:
            self.error = str(exc)

    def sample(self):
        used = self.nvml.nvmlDeviceGetMemoryInfo(self.handle).used
        self.peak_device_bytes = max(self.peak_device_bytes, used)

    def snapshot(self):
        self.sample()
        return {"peak_allocated_bytes": self.torch.cuda.max_memory_allocated(0),
            "peak_reserved_bytes": self.torch.cuda.max_memory_reserved(0),
            "peak_device_used_bytes_sampled": self.peak_device_bytes,
            "limit_bytes": VRAM_LIMIT_BYTES, "nvml_sampling_seconds": 0.05}

    def check(self):
        if self.error:
            raise RuntimeError("NVML monitoring failed: " + self.error)
        stats = self.snapshot()
        if max(stats[k] for k in ("peak_reserved_bytes", "peak_device_used_bytes_sampled")) >= VRAM_LIMIT_BYTES:
            raise MemoryBudgetExceeded(f"10 GB VRAM budget exceeded: {stats}")
        return stats

    def close(self):
        self.stop_event.set()
        self.thread.join(timeout=2)


class SQLGenerator:
    def __init__(self, adapter="./lora_sql_adapter_3epochs", max_context=2048, max_new_tokens=256):
        require_kaggle()
        import torch
        from peft import PeftConfig, PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
        assert torch.cuda.is_available()
        self.torch = torch
        self.max_context, self.max_new_tokens = max_context, max_new_tokens
        set_seed(3407)
        torch.cuda.set_per_process_memory_fraction(8_800_000_000 / torch.cuda.get_device_properties(0).total_memory, 0)
        torch.cuda.reset_peak_memory_stats(0)
        self.memory = MemoryMonitor(torch)
        config = PeftConfig.from_pretrained(adapter)
        assert config.r == 16 and config.lora_alpha == 16 and config.bias == "none"
        assert set(config.target_modules) == {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
        self.tokenizer = AutoTokenizer.from_pretrained(adapter)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION,
            device_map={"": "cuda:0"}, torch_dtype=torch.float16,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))
        self.model = PeftModel.from_pretrained(base, adapter, is_trainable=False).eval()
        assert self.model.is_loaded_in_4bit
        quant = self.model.config.quantization_config
        assert (quant.get("bnb_4bit_quant_type") if isinstance(quant, dict) else quant.bnb_4bit_quant_type) == "nf4"
        logging.info("Loaded NF4 base + final LoRA | memory=%s", self.memory.check())

    def __call__(self, question, db_id, schema):
        inputs = self.tokenizer.apply_chat_template(chat_messages(question, db_id, schema),
            tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        prompt_tokens = inputs["input_ids"].shape[1]
        if prompt_tokens + self.max_new_tokens > self.max_context:
            raise ValueError(f"Context limit: {prompt_tokens}+{self.max_new_tokens}>{self.max_context}; no silent truncation")
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        start = time.monotonic()
        with self.torch.no_grad():
            output = self.model.generate(**inputs, do_sample=False, temperature=0.0,
                num_beams=1, max_new_tokens=self.max_new_tokens, use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id)
        tokens = output[0, prompt_tokens:]
        answer = self.tokenizer.decode(tokens, skip_special_tokens=True)
        stats = {"seconds": time.monotonic() - start, "prompt_tokens": prompt_tokens,
            "generated_tokens": len(tokens), "hit_token_limit": len(tokens) == self.max_new_tokens}
        del inputs, output, tokens
        stats["memory"] = self.memory.check()
        return answer, stats


def verify_model(generator):
    """Actual GPU inference and native-template assertions before evaluation."""
    schema = "CREATE TABLE item(id INTEGER PRIMARY KEY);"
    messages = chat_messages("How many items?", "probe", schema)
    assert len(messages) == 2 and messages[0]["content"].endswith(schema)
    assert assistant_target("SELECT 1")["content"] == "```sql\nSELECT 1\n```"
    rendered = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    assert "<|im_start|>system" in rendered and "<|im_start|>assistant" in rendered
    answer, metadata = generator("How many items?", "probe", schema)
    assert answer.strip() and metadata["generated_tokens"] > 0
    generator.memory.check()
    logging.info("CHUNK 3 PASS | GPU generation=%r | memory=%s", answer, metadata["memory"])


In [ ]:
import hashlib
from notebook2.model import SQLGenerator, verify_model
adapter_files = list(Path('/kaggle/input').rglob('lora_sql_adapter_3epochs/adapter_model.safetensors'))
assert len(adapter_files) == 1
with adapter_files[0].open('rb') as stream:
    adapter_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
assert adapter_sha == '961d13a4c49a15b92e1a0801f72e86253f4c4f5f7665786e9e758fd6ad1f5a9a'
generator = SQLGenerator(str(adapter_files[0].parent))
verify_model(generator)
passed_chunks.append(3)


## Chunk 4: SQL Sanitizer & SQLite Execution Harness

Accept one SQL code block or one bare query. Remove comments outside quoted
strings without rewriting SQL spelling. SQLite opens existing files read-only;
an authorizer, five-second VM deadline, and 100,000-row cap limit execution.
Syntax validity is separate from missing-table/column execution errors.


In [ ]:
%%writefile /kaggle/working/notebook2/sql_engine.py
"""Chunk 4: lossless comment stripping and bounded, read-only SQLite scoring."""
from contextlib import closing
import logging
from pathlib import Path
import re
import sqlite3
import time

import pandas as pd
import sqlglot
from sqlglot import exp


def strip_comments(sql):
    """Remove comments outside quoted literals/identifiers, preserving SQL spelling.

    Re-rendering an AST can change result column labels (COUNT(*) vs count(*)),
    so keep the original query for strict Pandas equality with Notebook 1.
    """
    out, i, quote = [], 0, None
    while i < len(sql):
        char = sql[i]
        if quote:
            out.append(char)
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote and quote != "]":
                    out.append(sql[i + 1])
                    i += 1
                else:
                    quote = None
        elif char in "'\"`[":
            quote = "]" if char == "[" else char
            out.append(char)
        elif sql[i:i+2] == "--":
            end = sql.find("\n", i + 2)
            i = len(sql) if end == -1 else end
            out.append("\n")
            continue
        elif sql[i:i+2] == "/*":
            end = sql.find("*/", i + 2)
            if end == -1:
                raise ValueError("Unterminated SQL comment")
            i = end + 2
            out.append(" ")
            continue
        else:
            out.append(char)
        i += 1
    if quote:
        raise ValueError("Unterminated SQL quote")
    return "".join(out).strip()


def parse_readonly(sql):
    statements = [s for s in sqlglot.parse(sql, read="sqlite") if s is not None]
    if len(statements) != 1 or not isinstance(statements[0], exp.Query):
        raise ValueError("Expected exactly one SELECT/CTE query")
    # Block modifying CTEs as well as top-level mutations.
    if any(isinstance(node, (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop))
           for node in statements[0].walk()):
        raise ValueError("Modifying statements are forbidden")
    return statements[0]


def sanitize_sql(answer):
    blocks = re.findall(r"```(?:sql|sqlite)?\s*\n?(.*?)```", answer, re.I | re.S)
    if len(blocks) > 1 or ("```" in answer and not blocks):
        raise ValueError("Expected one complete SQL code block")
    sql = strip_comments(blocks[0] if blocks else answer)
    parse_readonly(sql)
    return sql


def execute_dataframe(path, sql, timeout=5.0, max_rows=100000):
    if timeout <= 0 or max_rows <= 0:
        raise ValueError("Positive timeout and row limit required")
    parse_readonly(sql)
    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ,
               sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}
    def authorize(action, arg1, arg2, db, source):
        if action == sqlite3.SQLITE_FUNCTION and (arg2 or "").lower() == "load_extension":
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY
    start = time.monotonic()
    with closing(sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True, timeout=timeout)) as conn:
        conn.execute("PRAGMA query_only=ON")
        conn.set_authorizer(authorize)
        conn.set_progress_handler(lambda: int(time.monotonic() - start > timeout), 1000)
        cursor = conn.execute(sql)
        rows = cursor.fetchmany(max_rows + 1)
        if len(rows) > max_rows:
            raise ValueError("Result row limit exceeded")
        return pd.DataFrame.from_records(rows, columns=[column[0] for column in cursor.description])


def score_prediction(path, answer, gold):
    result = {"sql": "", "syntax_valid": False, "execution_valid": False,
        "execution_match": False, "error_type": None, "error": None}
    try:
        result["sql"] = sanitize_sql(answer)
    except (ValueError, sqlglot.errors.SqlglotError) as exc:
        return {**result, "error_type": "invalid_sql", "error": str(exc)}
    result["syntax_valid"] = True
    try:
        predicted = execute_dataframe(path, result["sql"])
        result.update(execution_valid=True, execution_match=gold is not None and predicted.equals(gold),
                      result_shape=list(predicted.shape))
    except sqlite3.Error as exc:
        syntax_error = any(marker in str(exc).lower() for marker in
                           ("syntax error", "incomplete input", "unrecognized token"))
        result.update(syntax_valid=not syntax_error,
            error_type="sqlite_syntax" if syntax_error else "sqlite_execution", error=str(exc))
    except (ValueError, sqlglot.errors.SqlglotError) as exc:
        result.update(error_type="resource_or_query_limit", error=str(exc))
    return result


def verify_sql_engine():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "fixture.sqlite"
        with sqlite3.connect(path) as conn:
            conn.executescript("CREATE TABLE item(id INTEGER PRIMARY KEY, name TEXT);"
                               "INSERT INTO item VALUES (1,'a'),(2,'b');")
        gold = execute_dataframe(path, "SELECT count(*) FROM item")
        assert score_prediction(path, "```sql\nSELECT count(*) FROM item; -- done\n```", gold)["execution_match"]
        assert not score_prediction(path, "SELECT count(*) AS n FROM item", gold)["execution_match"]
        assert sanitize_sql("SELECT '--literal/*x*/' AS x -- note") == "SELECT '--literal/*x*/' AS x"
        assert not score_prediction(path, "SELECT FROM", gold)["syntax_valid"]
        missing = score_prediction(path, "SELECT * FROM nonexistent", gold)
        assert missing["syntax_valid"] and not missing["execution_valid"]
        for sql in ("DELETE FROM item", "SELECT 1; DROP TABLE item", "ATTACH DATABASE 'x' AS x",
                    "SELECT load_extension('x')", "SELECT 1 /* unclosed"):
            assert not score_prediction(path, sql, gold)["execution_valid"]
        assert not score_prediction(path, "```sql\nSELECT 1\n```\n```sql\nSELECT 2\n```", gold)["syntax_valid"]
        try:
            execute_dataframe(path, "SELECT * FROM item", max_rows=1)
            raise AssertionError("Row cap not enforced")
        except ValueError:
            pass
        try:
            execute_dataframe(path, "WITH RECURSIVE x(n) AS (SELECT 1 UNION ALL SELECT n+1 FROM x) SELECT sum(n) FROM x", timeout=.01)
            raise AssertionError("Timeout not enforced")
        except sqlite3.OperationalError as exc:
            assert "interrupted" in str(exc)
        assert execute_dataframe(path, "SELECT count(*) FROM item").equals(gold)
    logging.info("CHUNK 4 PASS | sanitation, strict equality, syntax/errors, read-only, timeout, row cap")


In [ ]:
from notebook2.sql_engine import verify_sql_engine
verify_sql_engine()
passed_chunks.append(4)


## Chunk 5: Spider Evaluation Pipeline & Comparative Analysis

Replay Notebook 1's validation snapshot and exact 100 indices. Verify every
database hash and the adapter hash. Index only schema DDL, retrieve three tables
per question, generate SQL once, and execute against the same gold query.
All selected examples remain in the denominator. Raw predictions, retrieved
tables, errors, timings and memory metrics are saved for auditing.

Top-3 retrieval can omit join bridge tables. No retries or gold-guided schema
selection are used. Comparative improvements include the new system prompt as
well as schema retrieval; this is not a controlled retrieval-only ablation.


In [ ]:
%%writefile /kaggle/working/notebook2/pipeline.py
"""Chunk 5: identical Spider samples, audited retrieval, EX and comparison exports."""
import hashlib
import json
import logging
from pathlib import Path
import sqlite3
import time

import pandas as pd
import sqlglot

from notebook2.model import MemoryBudgetExceeded, BASE_MODEL, BASE_REVISION
from notebook2.schemas import database_path
from notebook2.sql_engine import execute_dataframe, score_prediction

ADAPTER_SHA256 = "961d13a4c49a15b92e1a0801f72e86253f4c4f5f7665786e9e758fd6ad1f5a9a"


def sha256_file(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def discover_inputs():
    root = Path("/kaggle/input")
    adapters = list(root.rglob("lora_sql_adapter_3epochs/adapter_model.safetensors"))
    priors = list(root.rglob("qlora-spider-benchmark-100/run_provenance.json"))
    databases = list(root.rglob("concert_singer/concert_singer.sqlite"))
    assert len(adapters) == len(priors) == len(databases) == 1, (adapters, priors, databases)
    assert sha256_file(adapters[0]) == ADAPTER_SHA256, "Wrong final adapter"
    return adapters[0].parent, priors[0].parent, databases[0].parent.parent


def load_comparable_inputs(prior_root, db_root):
    prior_root = Path(prior_root)
    provenance = json.loads((prior_root / "run_provenance.json").read_text())
    summary = json.loads((prior_root / "spider_benchmark_100/summary.json").read_text())
    rows = json.loads((prior_root / "validation.json").read_text())
    indices = provenance["sample_indices"]
    previous = [json.loads(line) for line in (prior_root / "spider_benchmark_100/predictions.jsonl").read_text().splitlines()]
    assert len(indices) == len(previous) == 100 and len(set(indices)) == 100
    assert indices == summary["metadata"]["sample_indices"] == [r["index"] for r in previous]
    assert provenance["adapter_sha256"] == ADAPTER_SHA256
    assert summary["metric"] == "strict_pandas_dataframe_equals_single_database"
    assert summary["metadata"]["base_revision"] == BASE_REVISION
    assert summary["gold_errors"] == 0 and summary["metadata"]["arguments"]["schema"] == "none"
    for i, old in zip(indices, previous):
        assert (rows[i]["db_id"], rows[i]["question"], rows[i]["query"]) == (old["db_id"], old["question"], old["gold_sql"])
    for db_id in sorted({rows[i]["db_id"] for i in indices}):
        assert sha256_file(database_path(db_root, db_id)) == provenance["database_sha256"][db_id]
    for i in indices:
        execute_dataframe(database_path(db_root, rows[i]["db_id"]), rows[i]["query"])
    logging.info("Preflight PASS | 100 identical samples, 19 matching DB hashes, final adapter and base revision")
    return rows, indices, previous, summary, provenance


def aggregate(records):
    if not records:
        raise ValueError("Empty evaluation")
    n = len(records)
    return {"samples": n, "metric": "strict_pandas_dataframe_equals_single_database",
        "execution_accuracy_pct": 100 * sum(r["rag"]["execution_match"] for r in records) / n,
        "execution_validity_pct": 100 * sum(r["rag"]["execution_valid"] for r in records) / n,
        "syntax_validity_pct": 100 * sum(r["rag"]["syntax_valid"] for r in records) / n,
        "syntax_error_rate_pct": 100 * sum(not r["rag"]["syntax_valid"] for r in records) / n,
        "gold_errors": sum(r["gold_error"] is not None for r in records),
        "generation_errors": sum(r["rag"]["error_type"] == "generation" for r in records),
        "retrieval_errors": sum(r["rag"]["error_type"] == "retrieval" for r in records)}


def evaluate(rows, indices, db_root, index, generator, output, top_k=3):
    """SQL errors count as failures; resource-budget violations stop the run."""
    records = []
    with (Path(output) / "predictions.jsonl").open("w") as stream:
        for position, i in enumerate(indices, 1):
            row = rows[i]
            path = database_path(db_root, row["db_id"])
            record = {"index": i, "db_id": row["db_id"], "question": row["question"],
                "gold_sql": row["query"], "gold_error": None, "retrieval": [], "raw_output": "", "generation": {}}
            try:
                gold = execute_dataframe(path, row["query"])
            except (ValueError, sqlite3.Error, sqlglot.errors.SqlglotError) as exc:
                gold = None
                record["gold_error"] = str(exc)
            stage = "retrieval"
            try:
                schema, matches = index.retrieve(row["question"], row["db_id"], top_k)
                record["retrieval"] = matches
                stage = "generation"
                answer, metadata = generator(row["question"], row["db_id"], schema)
                record.update(raw_output=answer, generation=metadata)
                record["rag"] = score_prediction(path, answer, gold)
            except MemoryBudgetExceeded:
                raise
            except (ValueError, KeyError, RuntimeError) as exc:
                # Never turn CUDA OOM into 100 misleading ordinary SQL failures.
                if "out of memory" in str(exc).lower() or "NVML" in str(exc):
                    raise
                record["rag"] = {"sql": "", "syntax_valid": False, "execution_valid": False,
                    "execution_match": False, "error_type": stage, "error": str(exc)}
            records.append(record)
            stream.write(json.dumps(record) + "\n")
            stream.flush()
            logging.info("[%d/%d] db=%s tables=%s | EX=%s | error=%s", position, len(indices), row["db_id"],
                [m["metadata"]["table_name"] for m in record["retrieval"]],
                record["rag"]["execution_match"], record["rag"]["error"])
    return records


def save_results(records, previous, previous_summary, output, metadata):
    output = Path(output)
    summary = {**aggregate(records), "metadata": metadata}
    prior_error = previous_summary["finetuned"]["syntax_error_rate_pct"]
    summary["syntax_error_rate_reduction_vs_3epoch_pct"] = (
        100 * (prior_error - summary["syntax_error_rate_pct"]) / prior_error if prior_error else None)
    summary["ex_gain_vs_3epoch_percentage_points"] = summary["execution_accuracy_pct"] - previous_summary["finetuned"]["execution_accuracy_pct"]
    old = {r["index"]: r for r in previous}
    table = pd.DataFrame([{"Index": r["index"], "Database": r["db_id"], "Question": r["question"],
        "Gold SQL": r["gold_sql"], "Base Model SQL": old[r["index"]]["base"]["sql"],
        "Schema-Free 3-Epoch SQL": old[r["index"]]["finetuned"]["sql"], "RAG SQL": r["rag"]["sql"],
        "Base Execution Match": old[r["index"]]["base"]["execution_match"],
        "3-Epoch Execution Match": old[r["index"]]["finetuned"]["execution_match"],
        "RAG Execution Match": r["rag"]["execution_match"], "RAG Syntax Valid": r["rag"]["syntax_valid"],
        "Retrieved Tables": ", ".join(m["metadata"]["table_name"] for m in r["retrieval"]),
        "RAG Error": r["rag"]["error"]} for r in records])
    comparison = pd.DataFrame([
        {"Model": "Unadapted base / schema-free", "EX (%)": previous_summary["base"]["execution_accuracy_pct"], "Status": "Measured, same 100"},
        {"Model": "Notebook 1 / 60 steps / schema-free", "EX (%)": None, "Status": "Unmeasured; 15% is the 3-epoch result"},
        {"Model": "Notebook 1 / 3 epochs / schema-free", "EX (%)": previous_summary["finetuned"]["execution_accuracy_pct"], "Status": "Measured, same 100"},
        {"Model": "Notebook 2 / 3 epochs / schema RAG", "EX (%)": summary["execution_accuracy_pct"], "Status": "Measured, same 100"}])
    table.to_csv(output / "comparison.csv", index=False)
    comparison.to_csv(output / "performance_summary.csv", index=False)
    (output / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    (output / "comparison.html").write_text('<!doctype html><html><head><meta charset="utf-8">'
        '<title>Spider Schema RAG</title><style>body{font-family:system-ui}table{border-collapse:collapse}'
        'th,td{border:1px solid #bbb;padding:8px;vertical-align:top;max-width:450px;overflow-wrap:anywhere}'
        '</style></head><body><h1>Notebook 2: Schema-aware RAG</h1>'
        '<p>Strict Pandas equality on identical SQLite databases. Not official Spider test-suite accuracy. '
        'Column labels, dtypes and row/column order must match exactly.</p>' + comparison.to_html(index=False, escape=True)
        + table.to_html(index=False, escape=True) + '</body></html>')
    logging.info("PERFORMANCE\n%s", comparison.to_string(index=False))
    return summary


def verify_pipeline():
    """Small deterministic remote fixture: error accounting and metric denominator."""
    import tempfile
    class FixtureIndex:
        def retrieve(self, question, db_id, top_k):
            return "CREATE TABLE item(id INTEGER);", [{"metadata": {"db_id": db_id, "table_name": "item"}}]
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        (root / "fixture").mkdir()
        with sqlite3.connect(root / "fixture/fixture.sqlite") as conn:
            conn.executescript("CREATE TABLE item(id INTEGER); INSERT INTO item VALUES(1);")
        rows = [{"db_id": "fixture", "question": q, "query": "SELECT id FROM item"} for q in ("good", "bad")]
        generator = lambda question, db_id, schema: ("SELECT id FROM item" if question == "good" else "SELECT * FROM absent", {})
        records = evaluate(rows, [0, 1], root, FixtureIndex(), generator, root)
        result = aggregate(records)
        assert result["samples"] == 2 and result["execution_accuracy_pct"] == 50
        assert result["syntax_validity_pct"] == 100 and result["gold_errors"] == 0
        assert len((root / "predictions.jsonl").read_text().splitlines()) == 2
    logging.info("CHUNK 5 PASS | complete denominator, failed SQL continuation, JSONL output")


In [ ]:
import json, time, logging
from notebook2.pipeline import (discover_inputs, load_comparable_inputs, evaluate,
    save_results, verify_pipeline, sha256_file, BASE_MODEL, BASE_REVISION)
verify_pipeline()
passed_chunks.append(5)
assert passed_chunks == [1, 2, 3, 4, 5]
adapter, prior_root, db_root = discover_inputs()
rows, indices, previous, previous_summary, prior_provenance = load_comparable_inputs(prior_root, db_root)
source_hashes = {p.name: sha256_file(p) for p in Path('notebook2').glob('*.py') if p.name != '__init__.py'}
verification = {'passed_chunks': passed_chunks, 'execution_location': 'Kaggle',
    'source_sha256': source_hashes, 'versions': versions}
Path('/kaggle/working/verification.json').write_text(json.dumps(verification, indent=2))
started = time.monotonic()
for db_id in sorted({rows[i]['db_id'] for i in indices}):
    from notebook2.schemas import database_path
    schema_index.ingest(database_path(db_root, db_id), db_id)
records = evaluate(rows, indices, db_root, schema_index, generator, OUTPUT, top_k=3)
memory = generator.memory.check()
generator.memory.close()
metadata = {'sample_indices': indices, 'seed': 3407, 'top_k': 3, 'versions': versions,
    'base_model': BASE_MODEL, 'base_revision': BASE_REVISION, 'adapter_sha256': adapter_sha,
    'prior_provenance': prior_provenance, 'source_sha256': source_hashes,
    'max_context': 2048, 'max_new_tokens': 256, 'do_sample': False, 'temperature': 0.0,
    'seconds_indexing_and_evaluation': time.monotonic() - started, 'memory': memory,
    'embedding_model': 'BAAI/bge-small-en-v1.5', 'embedding_device': 'CPU',
    'metric_note': 'Strict labels/dtypes/row and column order; not official Spider test-suite accuracy'}
summary = save_results(records, previous, previous_summary, OUTPUT, metadata)
assert summary['samples'] == 100 and summary['gold_errors'] == 0
assert summary['generation_errors'] == 0 and summary['retrieval_errors'] == 0, summary
verification['evaluation_complete'] = True
verification['memory'] = memory
Path('/kaggle/working/verification.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(summary, indent=2), flush=True)
from IPython.display import display
import pandas as pd
display(pd.read_csv(OUTPUT / 'performance_summary.csv'))
display(pd.read_csv(OUTPUT / 'comparison.csv').head(10))
